# Gold Layer: AI Persona Narratives

Generates AI-written investment narratives for every neighbourhood × city × persona
combination using Snowflake Cortex (`mistral-large`).

Reads from `GOLD.BOROUGH_SUMMARY`, `GOLD.INVESTMENT_SCORES`, and `GOLD.REVIEW_THEMES`.
Writes one row per neighbourhood × city × persona to `GOLD.AI_OUTPUTS`.

**Depends on:** Run `gold_borough_summary.ipynb`, `gold_investment_scores.ipynb`,
and `gold_review_themes.ipynb` first.

**Output:** `AIRBNB_INVESTMENT_DB.GOLD.AI_OUTPUTS`

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
CITIES         = ['BRISTOL', 'LONDON', 'MANCHESTER']
DATABASE       = 'AIRBNB_INVESTMENT_DB'
GOLD_SCHEMA    = 'GOLD'
OUTPUT_TABLE   = 'AI_OUTPUTS'
CORTEX_MODEL   = 'mistral-large'
PROMPT_VERSION = 'v1'

PERSONAS = {
    'FIRST_TIME_BUYER': {
        'label':       'First-Time Buyer',
        'focus':       'affordability, value for money, low risk entry point',
        'score_col':   'score_first_time_buyer',
        'top_metrics': ['avg_price', 'avg_review_rating', 'avg_occupancy', 'pct_superhost'],
    },
    'EXPERIENCED_LANDLORD': {
        'label':       'Experienced Landlord',
        'focus':       'revenue maximisation, yield, occupancy rate',
        'score_col':   'score_experienced_landlord',
        'top_metrics': ['avg_revenue', 'avg_occupancy', 'avg_price', 'avg_review_rating'],
    },
    'PASSIVE_INCOME': {
        'label':       'Passive Income Seeker',
        'focus':       'stable occupancy, low management overhead, consistent returns',
        'score_col':   'score_passive_income',
        'top_metrics': ['avg_occupancy', 'avg_revenue', 'avg_review_rating', 'pct_superhost'],
    },
}

In [ ]:
def load_gold_tables(session, city):
    df_borough = session.sql(f'''
        SELECT * FROM {DATABASE}.{GOLD_SCHEMA}.BOROUGH_SUMMARY
        WHERE city = '{city}'
    ''').to_pandas()

    df_scores = session.sql(f'''
        SELECT neighbourhood_cleansed,
               AVG(investment_score)  AS avg_investment_score,
               COUNT(*)               AS listing_count,
               AVG(norm_price)        AS avg_norm_price,
               AVG(norm_occupancy)    AS avg_norm_occupancy,
               AVG(norm_revenue)      AS avg_norm_revenue,
               AVG(norm_rating)       AS avg_norm_rating
        FROM {DATABASE}.{GOLD_SCHEMA}.INVESTMENT_SCORES
        WHERE city = '{city}'
        GROUP BY neighbourhood_cleansed
    ''').to_pandas()

    df_themes = session.sql(f'''
        SELECT neighbourhood_cleansed, top_theme,
               pct_mentions_price, pct_mentions_cleanliness,
               pct_mentions_location, pct_mentions_checkin,
               avg_sentiment_score
        FROM {DATABASE}.{GOLD_SCHEMA}.REVIEW_THEMES
        WHERE city = '{city}'
    ''').to_pandas()

    df = df_borough.merge(df_scores, on='neighbourhood_cleansed', how='left')
    df = df.merge(df_themes, on='neighbourhood_cleansed', how='left')

    print(f'Loaded {len(df)} neighbourhoods for {city}')
    return df


def build_metrics_json(row, persona_key):
    persona = PERSONAS[persona_key]
    metrics = {
        'neighbourhood':             row['neighbourhood_cleansed'],
        'persona':                   persona['label'],
        'investment_score':          round(row[persona['score_col']], 2),
        'avg_price':                 round(row['avg_price'], 2),
        'avg_occupancy':             round(row['avg_occupancy'], 4),
        'avg_revenue':               round(row['avg_revenue'], 2),
        'avg_review_rating':         round(row['avg_review_rating'], 2),
        'listing_count':             int(row['listing_count']),
        'pct_superhost':             round(row['pct_superhost'], 2),
        'top_review_theme':          row['top_theme'],
        'pct_mentions_price':        row['pct_mentions_price'],
        'pct_mentions_cleanliness':  row['pct_mentions_cleanliness'],
        'pct_mentions_location':     row['pct_mentions_location'],
        'pct_mentions_checkin':      row['pct_mentions_checkin'],
    }
    return metrics


def build_prompt(metrics, persona_key):
    persona = PERSONAS[persona_key]

    system_prompt = f'''You are an expert Airbnb investment analyst.
You are advising a {persona['label']} who prioritises
{persona['focus']}.
Write a concise 3-paragraph investment analysis for the
neighbourhood provided.
Paragraph 1: Overall investment potential for this persona.
Paragraph 2: Key strengths based on the data.
Paragraph 3: Key risks or considerations for this persona.
Be specific, reference the actual numbers provided.
Do not use bullet points. Do not mention other personas.
Keep total response under 250 words.'''

    occ_pct      = round(metrics['avg_occupancy'] * 100, 1)
    rev_fmt      = f"{metrics['avg_revenue']:,.0f}"
    superhost    = round(metrics['pct_superhost'], 1)
    price_pct    = metrics['pct_mentions_price']
    location_pct = metrics['pct_mentions_location']

    user_prompt = f'''
Neighbourhood: {metrics['neighbourhood']}
Investment Score ({persona['label']}): {metrics['investment_score']}/100
Average Nightly Price: \u00a3{metrics['avg_price']}
Estimated Occupancy Rate: {occ_pct}%
Estimated Annual Revenue: \u00a3{rev_fmt}
Average Review Rating: {metrics['avg_review_rating']}/5
Total Listings: {metrics['listing_count']}
Superhost Percentage: {superhost}%
Top Review Theme: {metrics['top_review_theme']}
Price Mentions in Reviews: {price_pct:.1f}%
Location Mentions in Reviews: {location_pct:.1f}%
'''

    return system_prompt, user_prompt


def call_cortex(session, system_prompt, user_prompt):
    system_escaped = system_prompt.replace("'", "''")
    user_escaped   = user_prompt.replace("'", "''")

    result = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.COMPLETE(
            '{CORTEX_MODEL}',
            [
                {{'role': 'system', 'content': '{system_escaped}'}},
                {{'role': 'user',   'content': '{user_escaped}'}}
            ]
        ) AS response
    """).to_pandas()

    return result['RESPONSE'][0]


def generate_all_outputs(session, df_merged, city):
    rows = []
    for _, row in df_merged.iterrows():
        neighbourhood = row['neighbourhood_cleansed']
        for persona_key in PERSONAS:
            score_col = PERSONAS[persona_key]['score_col']
            metrics = build_metrics_json(row, persona_key)
            system_prompt, user_prompt = build_prompt(metrics, persona_key)

            try:
                ai_response = call_cortex(session, system_prompt, user_prompt)
            except Exception as e:
                ai_response = None
                print(f'  WARNING: Cortex call failed for {neighbourhood} / {persona_key}: {e}')

            rows.append({
                'city':                   city,
                'neighbourhood_cleansed': neighbourhood,
                'persona':                persona_key,
                'investment_score':       round(row[score_col], 2),
                'ai_narrative':           ai_response,
                'metrics_json':           str(metrics),
                'prompt_version':         PROMPT_VERSION,
                'model_used':             CORTEX_MODEL,
                'computed_at':            pd.Timestamp.now(),
            })
            print(f'  {neighbourhood} / {persona_key} \u2014 done')

    return pd.DataFrame(rows)


def write_gold(session, df):
    session.write_pandas(
        df,
        OUTPUT_TABLE,
        database=DATABASE,
        schema=GOLD_SCHEMA,
        overwrite=True,
        auto_create_table=True
    )
    print(f'Written {len(df)} rows to {GOLD_SCHEMA}.{OUTPUT_TABLE}')


def validate(session):
    df_val = session.sql(f'''
        SELECT city, persona,
               COUNT(*) AS neighbourhood_count,
               COUNT(ai_narrative) AS narratives_generated
        -- narratives_generated should equal neighbourhood_count
        -- any 0s mean Cortex calls failed — check warnings above
        FROM {DATABASE}.{GOLD_SCHEMA}.{OUTPUT_TABLE}
        GROUP BY city, persona
        ORDER BY city, persona
    ''').to_pandas()
    print(df_val)

In [ ]:
# --- Orchestration ---
all_dfs = []
for city in CITIES:
    print(f'\nProcessing {city}...')
    df_merged = load_gold_tables(session, city)
    print(f'  {len(df_merged)} neighbourhoods loaded')
    df_outputs = generate_all_outputs(session, df_merged, city)
    all_dfs.append(df_outputs)
    print(f'  {city} complete \u2014 {len(df_outputs)} rows generated')

df_final = pd.concat(all_dfs, ignore_index=True)
write_gold(session, df_final)
validate(session)

In [ ]:
df_val = session.sql(f'''
    SELECT city, persona,
           COUNT(*) AS neighbourhood_count,
           COUNT(ai_narrative) AS narratives_generated
    FROM {DATABASE}.{GOLD_SCHEMA}.{OUTPUT_TABLE}
    GROUP BY city, persona
    ORDER BY city, persona
''').to_pandas()
print(df_val)

## AI Persona Narratives Complete

`GOLD.AI_OUTPUTS` is ready for Streamlit consumption.
Each row represents one neighbourhood × city × persona with a Cortex-generated investment narrative.
If `narratives_generated` is less than `neighbourhood_count` for any group, check the warnings
printed during `generate_all_outputs` — those rows will have `ai_narrative = NULL`.